# AlphaZero-Style Chess RL Training

Based on Leela Chess Zero (Lc0) architecture and the research paper:
"Methods for analysing Leela Chess Zero"

**Key improvements over v2:**
- Monte Carlo Tree Search (MCTS) for lookahead
- Policy head (move probabilities) + Value head (position evaluation)
- WDL head (Win/Draw/Loss probabilities)
- Self-play with strength improvement
- Better endgame understanding through tree search


## 1. Setup & Dependencies

In [ ]:
!pip install python-chess torch numpy matplotlib -q

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import chess
import chess.pgn
import numpy as np
import json
from datetime import datetime
from pathlib import Path
import random
from collections import defaultdict
import matplotlib.pyplot as plt
import re
from copy import deepcopy

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Using device: {device}")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
drive_path = Path('/content/drive/MyDrive/chess_alphazero')
drive_path.mkdir(parents=True, exist_ok=True)
print(f"✓ Models saved to: {drive_path}")

## 3. Board Encoding (Lc0 style: 8×8×112 tensor)

In [ ]:
class BoardEncoder:
    """
    Encodes chess position as 8×8×112 tensor (Lc0 format):
    - Planes 0-103: 8 move history states (13 planes each = 8 piece types for each side + 1 side to move)
    - Planes 104-111: Game state metadata (castling rights, en passant, etc.)
    """
    
    def __init__(self, history_size=8):
        self.history_size = history_size
        self.piece_to_idx = {
            chess.PAWN: 0, chess.KNIGHT: 1, chess.BISHOP: 2, chess.ROOK: 3,
            chess.QUEEN: 4, chess.KING: 5
        }
    
    def encode_board(self, board, move_history=None):
        """
        Encode board and move history into 112-plane tensor.
        move_history: list of previous boards for context
        """
        tensor = torch.zeros(112, 8, 8, dtype=torch.float32)
        
        # Use move history if provided, else just current board
        boards = move_history[-self.history_size:] if move_history else [board]
        
        # Pad with initial position if history shorter than needed
        while len(boards) < self.history_size:
            boards.insert(0, chess.Board())
        
        # Planes 0-103: Piece positions (13 planes per timestep)
        for t, hist_board in enumerate(boards[-self.history_size:]):
            plane_offset = t * 13
            
            # Planes 0-5: White pieces (P, N, B, R, Q, K)
            for piece_type in range(1, 7):
                plane_idx = plane_offset + (piece_type - 1)
                for square in hist_board.pieces(piece_type, chess.WHITE):
                    row, col = square // 8, square % 8
                    tensor[plane_idx, row, col] = 1.0
            
            # Planes 6-11: Black pieces
            for piece_type in range(1, 7):
                plane_idx = plane_offset + 6 + (piece_type - 1)
                for square in hist_board.pieces(piece_type, chess.BLACK):
                    row, col = square // 8, square % 8
                    tensor[plane_idx, row, col] = 1.0
            
            # Plane 12: Side to move (1 if black's turn)
            if hist_board.turn == chess.BLACK:
                tensor[plane_offset + 12, :, :] = 1.0
        
        # Planes 104-107: Castling rights
        if board.has_kingside_castling_rights(chess.WHITE):
            tensor[104, :, :] = 1.0
        if board.has_queenside_castling_rights(chess.WHITE):
            tensor[105, :, :] = 1.0
        if board.has_kingside_castling_rights(chess.BLACK):
            tensor[106, :, :] = 1.0
        if board.has_queenside_castling_rights(chess.BLACK):
            tensor[107, :, :] = 1.0
        
        # Plane 108: Side to move (current)
        if board.turn == chess.BLACK:
            tensor[108, :, :] = 1.0
        
        # Plane 109: Halfmove clock (normalized to 0-1)
        tensor[109, :, :] = board.halfmove_clock / 100.0
        
        # Plane 111: Constant plane (all ones)
        tensor[111, :, :] = 1.0
        
        return tensor

encoder = BoardEncoder()
test_tensor = encoder.encode_board(chess.Board())
print(f"✓ Board tensor shape: {test_tensor.shape}")

## 4. Lc0-Style Network Architecture

In [ ]:
class ResidualBlock(nn.Module):
    """Residual block for deeper networks."""
    def __init__(self, filters):
        super().__init__()
        self.conv1 = nn.Conv2d(filters, filters, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(filters)
        self.conv2 = nn.Conv2d(filters, filters, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(filters)
    
    def forward(self, x):
        residual = x
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + residual
        out = torch.relu(out)
        return out


class ChessNetworkLc0(nn.Module):
    """
    Lc0-style network with:
    - Shared trunk (CNN)
    - Policy head: outputs move probabilities
    - Value head: outputs position evaluation (-1 to 1)
    - WDL head: outputs Win/Draw/Loss probabilities
    """
    
    def __init__(self, filters=128, blocks=6):
        super().__init__()
        self.filters = filters
        
        # Input layer: 112 planes -> filters
        self.input_conv = nn.Conv2d(112, filters, kernel_size=3, padding=1)
        self.input_bn = nn.BatchNorm2d(filters)
        
        # Residual blocks
        self.res_blocks = nn.Sequential(*[ResidualBlock(filters) for _ in range(blocks)])
        
        # ========== POLICY HEAD ==========
        # Outputs probability for each legal move
        self.policy_conv = nn.Conv2d(filters, 64, kernel_size=3, padding=1)
        self.policy_bn = nn.BatchNorm2d(64)
        self.policy_fc = nn.Linear(64 * 64, 4672)  # 64 src squares × 73 move types
        
        # ========== VALUE HEAD ==========
        # Outputs single value: position evaluation
        self.value_conv = nn.Conv2d(filters, 32, kernel_size=1)
        self.value_bn = nn.BatchNorm2d(32)
        self.value_fc1 = nn.Linear(32 * 64, 128)
        self.value_fc2 = nn.Linear(128, 1)  # Single output: evaluation
        
        # ========== WDL HEAD ==========
        # Outputs probabilities: Win, Draw, Loss
        self.wdl_conv = nn.Conv2d(filters, 32, kernel_size=1)
        self.wdl_bn = nn.BatchNorm2d(32)
        self.wdl_fc1 = nn.Linear(32 * 64, 128)
        self.wdl_fc2 = nn.Linear(128, 3)  # W, D, L
    
    def forward(self, x):
        # Shared trunk
        x = torch.relu(self.input_bn(self.input_conv(x)))
        x = self.res_blocks(x)
        
        # Policy head
        p = torch.relu(self.policy_bn(self.policy_conv(x)))
        p = p.view(p.size(0), -1)
        policy = torch.softmax(self.policy_fc(p), dim=1)
        
        # Value head
        v = torch.relu(self.value_bn(self.value_conv(x)))
        v = v.view(v.size(0), -1)
        v = torch.relu(self.value_fc1(v))
        value = torch.tanh(self.value_fc2(v))
        
        # WDL head
        w = torch.relu(self.wdl_bn(self.wdl_conv(x)))
        w = w.view(w.size(0), -1)
        w = torch.relu(self.wdl_fc1(w))
        wdl = torch.softmax(self.wdl_fc2(w), dim=1)
        
        return policy, value, wdl


net = ChessNetworkLc0(filters=128, blocks=6).to(device)
print(f"✓ Network parameters: {sum(p.numel() for p in net.parameters()):,}")

# Test forward pass
test_input = torch.randn(1, 112, 8, 8).to(device)
policy, value, wdl = net(test_input)
print(f"  Policy shape: {policy.shape}")
print(f"  Value shape: {value.shape}")
print(f"  WDL shape: {wdl.shape}")

## 5. Monte Carlo Tree Search (MCTS)

In [ ]:
class MCTSNode:
    """MCTS tree node."""
    
    def __init__(self, board, parent=None, move=None, prior=1.0):
        self.board = board.copy()
        self.parent = parent
        self.move = move  # Move that led to this node
        self.prior = prior  # Prior probability from policy head
        
        self.children = {}
        self.visit_count = 0
        self.value_sum = 0.0
        self.wdl_sum = np.zeros(3)  # W, D, L counts
    
    def ucb(self, c_puct=1.0):
        """Upper Confidence Bound for Trees (UCT) formula from AlphaZero."""
        if self.visit_count == 0:
            return float('inf')
        
        exploit = self.value_sum / self.visit_count
        parent_visits = self.parent.visit_count if self.parent else 1
        explore = self.prior * np.sqrt(parent_visits) / (1 + self.visit_count)
        
        return exploit + c_puct * explore
    
    def is_terminal(self):
        return self.board.is_game_over()


class MCTS:
    """Monte Carlo Tree Search with neural network guidance."""
    
    def __init__(self, network, encoder, c_puct=1.0, num_simulations=100):
        self.network = network
        self.encoder = encoder
        self.c_puct = c_puct
        self.num_simulations = num_simulations
    
    def select_action(self, board, temperature=1.0):
        """Run MCTS and return best move."""
        root = MCTSNode(board)
        
        for _ in range(self.num_simulations):
            self._simulate(root)
        
        # Select move with highest visit count
        if not root.children:
            # No moves found, return random legal move
            return random.choice(list(board.legal_moves)), root
        
        best_move = max(root.children.keys(), 
                       key=lambda m: root.children[m].visit_count)
        
        return best_move, root
    
    def _simulate(self, node):
        """One simulation: select → expand → evaluate → backup."""
        # Selection
        while not node.is_terminal() and node.children:
            best_move = max(node.children.keys(), key=lambda m: node.children[m].ucb(self.c_puct))
            node = node.children[best_move]
        
        # Expand if not terminal
        if not node.is_terminal():
            self._expand(node)
        
        # Evaluate with network
        value, wdl = self._evaluate_position(node.board)
        
        # Backup
        self._backup(node, value, wdl)
    
    def _expand(self, node):
        """Expand node by evaluating all legal moves."""
        board_tensor = self.encoder.encode_board(node.board).unsqueeze(0).to(device)
        
        with torch.no_grad():
            policy, _, _ = self.network(board_tensor)
            policy = policy[0].cpu().numpy()
        
        legal_moves = list(node.board.legal_moves)
        
        for move in legal_moves:
            # Map move to policy index (simplified: use move index)
            move_idx = min(hash(move) % len(policy), len(policy) - 1)
            prior = policy[move_idx]
            
            new_board = node.board.copy()
            new_board.push(move)
            child = MCTSNode(new_board, parent=node, move=move, prior=prior)
            node.children[move] = child
    
    def _evaluate_position(self, board):
        """Evaluate position using value and WDL heads."""
        board_tensor = self.encoder.encode_board(board).unsqueeze(0).to(device)
        
        with torch.no_grad():
            _, value, wdl = self.network(board_tensor)
            value = value[0].item()
            wdl = wdl[0].cpu().numpy()
        
        # Convert WDL to single value if game is over
        if board.is_game_over():
            result = board.result()
            if result == '1-0':
                return 1.0, np.array([1, 0, 0])
            elif result == '0-1':
                return -1.0, np.array([0, 0, 1])
            else:
                return 0.0, np.array([0, 1, 0])
        
        return value, wdl
    
    def _backup(self, node, value, wdl):
        """Backup value to root."""
        while node is not None:
            node.visit_count += 1
            node.value_sum += value
            node.wdl_sum += wdl
            node = node.parent

print("✓ MCTS implemented")

## 6. Self-Play Game

In [ ]:
def play_game(network, encoder, num_simulations=50, max_moves=150):
    """
    Play one game with MCTS.
    Returns: winner (0=white, 1=black, 2=draw), game_moves
    """
    mcts = MCTS(network, encoder, num_simulations=num_simulations)
    board = chess.Board()
    game_moves = []
    move_list = []
    
    for move_num in range(max_moves):
        if board.is_game_over():
            result = board.result()
            winner = 0 if result == '1-0' else (1 if result == '0-1' else 2)
            pgn = _create_pgn(move_list, result)
            return winner, game_moves, pgn
        
        # Select move with MCTS
        move, root = mcts.select_action(board)
        
        # Record state before move
        board_tensor = encoder.encode_board(board)
        
        # Make move
        board.push(move)
        move_list.append(move)
        
        # Store game data
        game_moves.append({
            'board': board_tensor,
            'move': move,
            'root': root,
            'mcts_probs': {m: root.children[m].visit_count for m in root.children}
        })
    
    # Max moves reached = draw
    pgn = _create_pgn(move_list, '*')
    return 2, game_moves, pgn


def _create_pgn(moves, result):
    """Create PGN string from move list."""
    game = chess.pgn.Game()
    node = game
    
    for move in moves:
        node = node.add_variation(move)
    
    game.headers["Event"] = "AlphaZero Self-Play"
    game.headers["White"] = "ChessNet"
    game.headers["Black"] = "ChessNet"
    game.headers["Result"] = result
    game.headers["Date"] = datetime.now().strftime("%Y.%m.%d")
    
    return str(game)

print("✓ Self-play game implemented")

## 7. Training Function

In [ ]:
def train_on_game(network, optimizer, game_moves, winner, encoder, learning_rate=0.001):
    """
    Train network on one game using game moves and result.
    
    Loss = policy_loss + value_loss + wdl_loss
    """
    network.train()
    total_loss = 0.0
    
    for i, move_data in enumerate(game_moves):
        board_tensor = move_data['board'].unsqueeze(0).to(device)
        move = move_data['move']
        root = move_data['root']
        
        # Get network prediction
        policy, value, wdl = network(board_tensor)
        
        # ========== POLICY TARGET ==========
        # Target is MCTS visit distribution
        mcts_probs = np.zeros(4672)
        total_visits = sum(root.children.values() if isinstance(root.children, dict) else [n.visit_count for n in root.children.values()])
        
        if total_visits > 0:
            for move_key, child in (root.children.items() if isinstance(root.children, dict) else [(m, root.children[m]) for m in root.children]):
                move_idx = min(hash(move_key) % len(mcts_probs), len(mcts_probs) - 1)
                mcts_probs[move_idx] = child.visit_count / total_visits
        
        policy_target = torch.tensor(mcts_probs, dtype=torch.float32).unsqueeze(0).to(device)
        
        # ========== VALUE TARGET ==========
        # Target is game outcome from this position
        if winner == 0:  # White won
            value_target = 1.0 if i % 2 == 0 else -1.0  # Flip perspective every move
        elif winner == 1:  # Black won
            value_target = -1.0 if i % 2 == 0 else 1.0
        else:  # Draw
            value_target = 0.0
        
        value_target = torch.tensor([[value_target]], dtype=torch.float32).to(device)
        
        # ========== WDL TARGET ==========
        if winner == 0:
            wdl_target = torch.tensor([[1, 0, 0]], dtype=torch.float32).to(device) if i % 2 == 0 else torch.tensor([[0, 0, 1]], dtype=torch.float32).to(device)
        elif winner == 1:
            wdl_target = torch.tensor([[0, 0, 1]], dtype=torch.float32).to(device) if i % 2 == 0 else torch.tensor([[1, 0, 0]], dtype=torch.float32).to(device)
        else:
            wdl_target = torch.tensor([[0, 1, 0]], dtype=torch.float32).to(device)
        
        # ========== LOSSES ==========
        policy_loss = -torch.sum(policy_target * torch.log(policy + 1e-8))
        value_loss = torch.mean((value - value_target) ** 2)
        wdl_loss = -torch.sum(wdl_target * torch.log(wdl + 1e-8))
        
        loss = policy_loss + value_loss + wdl_loss
        
        # Backprop
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(network.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    network.eval()
    return total_loss / len(game_moves) if game_moves else 0

print("✓ Training function implemented")

## 8. Save/Load Models

In [ ]:
def save_checkpoint(network, optimizer, episode, drive_path):
    """Save model checkpoint."""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    checkpoint = {
        'episode': episode,
        'model_state': network.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'timestamp': timestamp
    }
    
    path = drive_path / f'checkpoint_ep{episode}_{timestamp}.pt'
    torch.save(checkpoint, path)
    return path


def load_checkpoint(checkpoint_path):
    """Load model from checkpoint."""
    checkpoint = torch.load(checkpoint_path, map_location=device)
    network = ChessNetworkLc0(filters=128, blocks=6).to(device)
    optimizer = optim.Adam(network.parameters(), lr=0.001)
    
    network.load_state_dict(checkpoint['model_state'])
    optimizer.load_state_dict(checkpoint['optimizer_state'])
    
    return network, optimizer, checkpoint['episode']

print("✓ Checkpoint I/O implemented")

## 9. Training Statistics

In [ ]:
class TrainingStats:
    def __init__(self, drive_path):
        self.drive_path = drive_path
        self.stats_file = drive_path / 'training_stats.json'
        self.stats = {
            'total_games': 0,
            'white_wins': 0,
            'black_wins': 0,
            'draws': 0,
            'game_lengths': [],
            'loss_history': [],
            'recent_games': []
        }
        self.load()
    
    def load(self):
        if self.stats_file.exists():
            with open(self.stats_file) as f:
                self.stats = json.load(f)
    
    def record(self, winner, num_moves, loss):
        self.stats['total_games'] += 1
        self.stats['game_lengths'].append(num_moves)
        self.stats['loss_history'].append(loss)
        
        if winner == 0:
            self.stats['white_wins'] += 1
            self.stats['recent_games'].append('W')
        elif winner == 1:
            self.stats['black_wins'] += 1
            self.stats['recent_games'].append('B')
        else:
            self.stats['draws'] += 1
            self.stats['recent_games'].append('D')
        
        if len(self.stats['recent_games']) > 50:
            self.stats['recent_games'] = self.stats['recent_games'][-50:]
    
    def save(self):
        with open(self.stats_file, 'w') as f:
            json.dump(self.stats, f, indent=2)
    
    def print_summary(self):
        total = self.stats['total_games']
        if total == 0:
            return
        
        w = self.stats['white_wins']
        b = self.stats['black_wins']
        d = self.stats['draws']
        avg_len = np.mean(self.stats['game_lengths'][-20:]) if self.stats['game_lengths'] else 0
        avg_loss = np.mean(self.stats['loss_history'][-20:]) if self.stats['loss_history'] else 0
        
        print(f"  White: {w:3d} ({100*w/total:5.1f}%) | "
              f"Black: {b:3d} ({100*b/total:5.1f}%) | "
              f"Draw: {d:3d} ({100*d/total:5.1f}%) | "
              f"AvgLen: {avg_len:.1f} | "
              f"AvgLoss: {avg_loss:.4f}")

stats = TrainingStats(drive_path)
print("✓ Stats tracker initialized")

## 10. Main Training Loop

In [ ]:
# Configuration
NUM_EPISODES = 100  # Games to play
SAVE_INTERVAL = 20
MCTS_SIMULATIONS = 50  # Fewer simulations for speed
LEARNING_RATE = 0.001

# Auto-load checkpoint if exists
checkpoints = sorted(drive_path.glob('checkpoint_ep*.pt'))
start_episode = 0

if checkpoints:
    print(f"📦 Loading checkpoint: {checkpoints[-1].name}")
    net, optimizer, start_episode = load_checkpoint(checkpoints[-1])
    print(f"✓ Resumed from episode {start_episode}")
    print(f"\nPrevious training stats:")
    stats.print_summary()
else:
    net = ChessNetworkLc0(filters=128, blocks=6).to(device)
    optimizer = optim.Adam(net.parameters(), lr=LEARNING_RATE)
    net.eval()
    print("ℹ️  Starting fresh training")

print(f"\n🎯 Training {NUM_EPISODES} episodes (AlphaZero-style)")
print(f"   MCTS sims: {MCTS_SIMULATIONS}")
print(f"   Learning rate: {LEARNING_RATE}")
print(f"   Save interval: {SAVE_INTERVAL}\n")

last_pgn = None

try:
    for episode in range(start_episode, start_episode + NUM_EPISODES):
        # Play game with MCTS
        winner, game_moves, pgn = play_game(net, encoder, num_simulations=MCTS_SIMULATIONS)
        num_moves = len(game_moves)
        last_pgn = pgn
        
        # Train on game
        loss = train_on_game(net, optimizer, game_moves, winner, encoder, LEARNING_RATE)
        
        # Record stats
        stats.record(winner, num_moves, loss)
        
        # Progress report
        if (episode + 1) % 10 == 0:
            print(f"Episode {episode + 1:3d} | Moves: {num_moves:3d} | Loss: {loss:.4f} | ", end='')
            stats.print_summary()
        
        # Checkpoint
        if (episode + 1) % SAVE_INTERVAL == 0:
            save_checkpoint(net, optimizer, episode + 1, drive_path)
            stats.save()
            print(f"\n💾 Checkpoint saved at episode {episode + 1}\n")
    
    print("\n" + "="*80)
    print("✅ TRAINING COMPLETE!")
    print("="*80)
    stats.print_summary()
    
    save_checkpoint(net, optimizer, start_episode + NUM_EPISODES, drive_path)
    stats.save()
    print(f"\n📍 Final models saved to: {drive_path}")

except KeyboardInterrupt:
    print("\n⚠️  Training interrupted")
    save_checkpoint(net, optimizer, episode, drive_path)
    stats.save()
    print("✓ Models saved")

print("\n" + "="*80)
print("📋 LAST GAME (PGN):")
print("="*80)
if last_pgn:
    print(last_pgn)
else:
    print("No game played")

## 11. Analysis & Visualization

In [ ]:
# Load stats
with open(drive_path / 'training_stats.json') as f:
    final_stats = json.load(f)

if final_stats['total_games'] > 10:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Win distribution
    ax = axes[0, 0]
    labels = ['White Wins', 'Black Wins', 'Draws']
    sizes = [final_stats['white_wins'], final_stats['black_wins'], final_stats['draws']]
    colors = ['#FF6B6B', '#4ECDC4', '#95E1D3']
    ax.pie(sizes, labels=labels, autopct='%1.1f%%', colors=colors, startangle=90)
    ax.set_title(f'Win Distribution ({final_stats["total_games"]} games)')
    
    # Loss over time
    ax = axes[0, 1]
    if final_stats['loss_history']:
        losses = final_stats['loss_history']
        ax.plot(losses, marker='o', linewidth=1, markersize=3, alpha=0.7)
        ax.set_xlabel('Episode')
        ax.set_ylabel('Training Loss')
        ax.set_title('Loss Over Time')
        ax.grid(True, alpha=0.3)
    
    # Game lengths
    ax = axes[1, 0]
    if final_stats['game_lengths']:
        ax.hist(final_stats['game_lengths'], bins=30, color='skyblue', edgecolor='black', alpha=0.7)
        ax.set_xlabel('Moves')
        ax.set_ylabel('Frequency')
        avg_len = np.mean(final_stats['game_lengths'])
        ax.set_title(f'Game Length Distribution (avg: {avg_len:.1f})')
        ax.grid(True, alpha=0.3, axis='y')
    
    # Recent games cumulative
    ax = axes[1, 1]
    recent = final_stats['recent_games'][-50:]
    w_cum = np.cumsum([1 if x == 'W' else 0 for x in recent])
    b_cum = np.cumsum([1 if x == 'B' else 0 for x in recent])
    d_cum = np.cumsum([1 if x == 'D' else 0 for x in recent])
    
    ax.plot(w_cum, label='White', marker='o')
    ax.plot(b_cum, label='Black', marker='s')
    ax.plot(d_cum, label='Draw', marker='^')
    ax.set_xlabel('Games (last 50)')
    ax.set_ylabel('Cumulative Count')
    ax.set_title('Recent Performance')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(drive_path / 'training_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✓ Analysis saved to training_analysis.png")
else:
    print("Not enough games for visualization")

## 12. Quick Evaluation

In [ ]:
net.eval()

print("\n🎮 Running evaluation games...\n")
eval_results = {'white': 0, 'black': 0, 'draw': 0}

for i in range(3):
    winner, _, pgn = play_game(net, encoder, num_simulations=100, max_moves=150)
    
    if winner == 0:
        status = "⚪ White wins"
        eval_results['white'] += 1
    elif winner == 1:
        status = "⚫ Black wins"
        eval_results['black'] += 1
    else:
        status = "🤝 Draw"
        eval_results['draw'] += 1
    
    print(f"Game {i+1}: {status}")

print(f"\n📊 Eval Results: W:{eval_results['white']} B:{eval_results['black']} D:{eval_results['draw']}")